# WALLABY team release

This notebook helps WALLABY admin release source and products to VOSpace and YouCAT (CADC services).

---

In [ ]:
import os
import shutil
import getpass
import requests
import getpass

import pyvo as vo
from pyvo.auth import authsession, securitymethods

import numpy as np
from astropy.io import ascii
from astropy.wcs import WCS
from astropy.io.votable import from_table, parse_single_table
from astropy.table import vstack

import vos
from cadcutils import net
from cadctap import CadcTapClient

### Authenticate

<span style="font-weight: bold; color: #FF0000;">⚠ Update the cell below with your username and enter your password</span>

In [ ]:
# Enter WALLABY user username and password

username = 'wallaby_user'
password = getpass.getpass('Enter your password')

In [ ]:
# Connect with TAP service

URL = "https://wallaby.aussrc.org/tap"
auth = vo.auth.AuthSession()
auth.add_security_method_for_url(URL, vo.auth.securitymethods.BASIC)
auth.credentials.set_password(username, password)
tap = vo.dal.TAPService(URL, session=auth)

---

# Create release folder

<span style="font-weight: bold; color: #FF0000;">⚠ Update the cell below. Add tags to the list for release, and update `release_name` variable</span>

In [ ]:
# List of tags
tags = ['WALLABY']

# Release name
release_name = "Test"
release_name = release_name.replace(' ', '_')
os.makedirs(release_name, exist_ok=True)

---

# Detection Catalog

## Create VOTable

In [ ]:
# Retrieve catalog as Astropy table

query = """SELECT d.*, ivo_string_agg(t.name || ': ' || t.description, '; ') AS tags, ivo_string_agg(c.comment, '; ') AS comments
        FROM wallaby.detection d
        LEFT JOIN wallaby.tag_detection td ON d.id = td.detection_id 
        LEFT JOIN wallaby.tag t ON t.id = td.tag_id
        LEFT JOIN wallaby.comment c ON d.id = c.detection_id
        WHERE t.name IN ('Internal Data Release', '$TAG_NAME')
        GROUP BY d.id"""

In [ ]:
table = None
for idx, tag_name in enumerate(tags):
    q = query.replace('$TAG_NAME', tag_name)
    result = tap.search(q)
    if idx == 0:
        table = result.to_table()
        table['SRCTR'] = tag_name.replace(' ', '_').replace('DR', 'TR')
    else:
        new_table = result.to_table()
        new_table['SRCTR'] = tag_name.replace(' ', '_').replace('DR', 'TR')
        table = vstack([table, new_table])

## Modifying the catalog

There are some additional columns and calculated properties that are required for the release. The column metadata (e.g. UCDs, units, description etc as required to conform with VO standards) also need to be included for these additional columns. These include:

| Column | Description |
| --- | --- |
| `qflag` |  |
| `kflag` | column to indicate whether or not there is a kinematic model associated with the detection |
| `team_release` | Column with the release name |
| `f_sum_corr` | |
| `err_f_sum_corr` | |
| `dist_h` | |
| `log_m_hi_corr` | Uses `v_est` and `dist_est` which are calculated properties |

In [ ]:
# Table corrections

rest_freq = 1.42040575179E+09
c = 2.9979245e8
H0 = 70.0

write_table = table.copy()
write_table['name'] = table['source_name']
write_table['qflag'] = table['flag']
write_table['kflag'] = np.zeros(len(table['flag']))
write_table['team_release'] = release_name
write_table['f_sum_corr'] = table['f_sum'] / 10.0 ** (0.0285 * np.log10(table['f_sum'])**3.0 -0.439 * np.log10(table['f_sum'])**2.0 + 2.294 * np.log10(table['f_sum']) - 4.097)
write_table['err_f_sum_corr'] = table['err_f_sum'] / table['f_sum'] * write_table['f_sum_corr']
write_table['v_est'] = ((rest_freq - table['freq']) / table['freq'] * c / 1000.0)
write_table['dist_h'] = write_table['v_est'] / H0
write_table['log_m_hi'] = np.log10(49.7 * write_table['dist_h']**2.0 * table['f_sum'])
write_table['log_m_hi_corr'] = np.log10(49.7 * write_table['dist_h']**2.0 * write_table['f_sum_corr'])

In [ ]:
# Create column for linked product files

write_table['product_link'] = [f'https://www.canfar.net/storage/vault/list/WALLABY/detections/{sn.replace(' ', '_')}' for sn in table['source_name']]

In [ ]:
# Remove certain columns from the astropy table

write_table.remove_columns(['id', 'run_id', 'instance_id', 'access_url', 'access_format', 'source_name', 'flag', 'v_est', 'l', 'b', 'v_rad', 'v_opt', 'v_app', 'tags', 'SRCTR'])
votable = from_table(write_table)

In [ ]:
# Update derived quantity columns of votable

f_sum_corr_field = votable.get_field_by_id('f_sum_corr')
f_sum_corr_field.ucd = "phot.flux;meta.main"
f_sum_corr_field.unit = "Jy*Hz"
f_sum_corr_field.description = "The integrated flux within 3D source mask statistically corrected to match single dish observations"

err_f_sum_corr_field = votable.get_field_by_id('err_f_sum_corr')
err_f_sum_corr_field.ucd = "stat.error;phot.flux"
err_f_sum_corr_field.unit = "Jy*Hz"
err_f_sum_corr_field.description = "Statistical uncertainty of the single dish corrected integrated flux"

dist_h_field = votable.get_field_by_id('dist_h')
dist_h_field.ucd = "pos.distance"
dist_h_field.unit = "Mpc"
dist_h_field.description = "Local Hubble distance derived from the barycentric source frequency"

log_m_hi_field = votable.get_field_by_id('log_m_hi')
log_m_hi_field.ucd = "phys.mass"
log_m_hi_field.unit = "log10(Msol)"
log_m_hi_field.description = "The estimated log10 mass of the cube using f_sum and freq"

log_m_hi_corr_field = votable.get_field_by_id('log_m_hi_corr')
log_m_hi_corr_field.ucd = "phys.mass"
log_m_hi_corr_field.unit = "log10(Msol)"
log_m_hi_corr_field.description = "The estimated log10 mass of the cube using f_sum_corr and freq"

qflag_field = votable.get_field_by_id('qflag')
qflag_field.datatype = "double"
qflag_field.ucd = "meta.code.qual"
qflag_field.description = "Quality flag"

kflag_field = votable.get_field_by_id('kflag')
kflag_field.datatype = "double"
kflag_field.ucd = "meta.code"
kflag_field.description = "Kinematic model flag"

comments_field = votable.get_field_by_id('comments')
comments_field.datatype = "char"
comments_field.ucd = "meta.note"
comments_field.description = "Comments on individual sources"

team_release_field = votable.get_field_by_id('team_release')
team_release_field.datatype = "char"
team_release_field.ucd = "meta.dataset;meta.main"
team_release_field.description = "Internal team release identifier"

In [ ]:
# Update all table columns unicode char to char

product_link_field = votable.get_field_by_id('product_link')
product_link_field.datatype = 'char'

In [ ]:
# Download catalog table

votable.version = '1.3'
votable_filename = os.path.join(release_name, 'catalog.xml')
votable.to_xml(votable_filename)

# Tests

These tests verify data intergrity before removing existing catalogs and updating product files. These should pass.

In [ ]:
raise Exception('Tests need to pass')

## Upload to YouCAT

In [ ]:
# Upload catalog to YouCAT (delete table if exists)

table_name = 'wallaby.detection'
subject = net.Subject(certificate='/Users/she393/.ssl/cadcproxy.pem')
client = CadcTapClient(subject, resource_id='ivo://cadc.nrc.ca/youcat')

# delete if exists
try:
    client.delete_table(table_name)
except Exception as e:
    print(e)

# create and upload
client.create_table(table_name, votable_filename, 'VOTable')

---

# Products

Create product files in folders by source name. Upload to VOSpace.

In [ ]:
# useful function for downloading table products (requires authentication)

def download_products(row, products_filename, chunk_size=8192):
    """Download products for a row of the table (a detection entry)
    
    """
    name = row['source_name']
    access_url = row['access_url']
    votable = parse_single_table(access_url)
    product_table = votable.to_table()
    url = product_table[product_table['description'] == 'SoFiA-2 Detection Products'][0]['access_url']
    with requests.get(url, auth=(username, password), stream=True) as r:
        r.raise_for_status()
        with open(products_filename, 'wb') as f:
            for chunk in r.iter_content(chunk_size=chunk_size):
                f.write(chunk)
    print(f'Download completed for {name}')
    return

def download_table_products(table, directory, chunk_size=8192):
    """Download WALLABY products from ADQL queried table

    """
    if not os.path.exists(directory):
        os.mkdir(directory)
    print(f'Saving products to {directory}')
    for row in table:
        name = row['source_name']
        products_filename = os.path.join(directory, f'{name}.tar')
        download_products(row, products_filename, chunk_size)
    print('Downloads complete')
    return

In [ ]:
# Write output products for a source

download_table_products(table, release_name)

In [ ]:
# Update product files

import tarfile
import glob
from astropy.io import fits
import matplotlib.pyplot as plt

In [ ]:
def pixel_hdu(ra, dec, frequency):
    """Create dummy pixel hdu element (TBA)
    
    """
    return

def spectrum_to_fits(f_in, f_cube, f_out, ra, dec, frequency):
    """Convert the SoFiA-2 output spectrum .txt file to a .fits file for public release
    Writes a HDUList object with:
        - dummy pixel (fits metadata from the accompanying _cube.fits file
        - fits binary table with spectra.txt content
        
    """
    # read spectrum and construct binary table
    channels, freq, flux_density, pixels = np.loadtxt(f_in, skiprows=38, unpack=True, usecols=[0,1,2,3])    
    channels_col = fits.Column(name='Channel', format='D', array=channels.astype('int'), unit='')
    freq_col = fits.Column(name='Frequency', format='E', array=freq, unit='Hz')
    flux_density_col = fits.Column(name='Flux density', format='E', array=flux_density, unit='Jy')
    pixel_col = fits.Column(name='Pixels', format='D', array=pixels.astype('int'), unit='')
    fits_table = fits.BinTableHDU.from_columns([channels_col, freq_col, flux_density_col, pixel_col])
    
    # construct dummy image hdu
    keys = ['OBJECT', 'CDELT1', 'CDELT2', 'CDELT3', 'CTYPE1', 'CTYPE2', 'CTYPE3', 'ORIGIN', 'EQUINOX', 'LONPOLE', 'LATPOLE', 'SRCVERS', 'SRCTR']
    # keys += ['SBID']
    with fits.open(f_cube, mode='readonly') as hdu_cube:
        header_cube = hdu_cube[0].header
    hdu = fits.PrimaryHDU()
    header = hdu.header
    hdu.data = np.array([[[0]]]).astype('int16')
    header['CRPIX1'] = 0
    header['CRPIX2'] = 0
    header['CRPIX3'] = 0
    header['CUNIT1'] = 'deg'
    header['CUNIT2'] = 'deg'
    header['CUNIT3'] = 'Hz'
    header['CRVAL1'] = ra
    header['CRVAL2'] = dec
    header['CRVAL3'] = frequency
    header['SPECSYS'] = 'BARYCENT'
    header['RADESYS'] = 'FK5'
    for k in keys:
        header.set(k, header_cube[k])    

    # construct hdulist and write to file
    hdu_list = fits.HDUList([hdu, fits_table])
    hdu_list.writeto(f_out, overwrite=True, output_verify='fix')
    return

def mom0_to_png(data, f_out):
    """Create plot of mom0 map as png file for archive cutouts

    """
    plt.imshow(data)
    plt.axis('off')
    plt.savefig(f_out, bbox_inches='tight', pad_inches=0)
    plt.close()
    return

In [ ]:
# Update all product files

product_tarfiles = glob.glob(os.path.join(release_name, '*.tar'))
product_tarfiles.sort()
product_files = []

# Extract
for f in product_tarfiles:
    filename = f.replace('.tar', '')
    filename = filename.replace(' ', '_')
    product_files.append(filename)
    with tarfile.open(f) as tf:
        tf.extractall(path=filename)

In [ ]:
# Update product files
for idx_pf, pf_raw in enumerate(product_files):
    print(f'[{idx_pf}] Folder {pf_raw} [{idx_pf + 1}/{len(product_files)}]')
    fits_files = glob.glob(os.path.join(pf_raw, '*.fits'))
    source_name_file = pf_raw.split('/')[1]
    for idx_ff, ff in enumerate(fits_files):
        source_name = ff.split('/')[1].replace('_', ' ')
        with fits.open(ff, mode='update') as hdul:
            header = hdul[0].header
            # NOTE: DATE card?
            header['SRCVERS'] = header['ORIGIN']  # Get SoFiA version from ORIGIN header
            header['SRCTR'] = release_name
            header['OBJECT'] = source_name
            hdul.flush()

            # Download moment 0 map figure
            if 'mom0.fits' in ff:
                data = hdul[0].data
                mom0_png = ff.replace('.fits', '.png')
                mom0_to_png(data, mom0_png)

    # Update spectra
    spectra_files = glob.glob(os.path.join(pf_raw, '*spec.txt'))
    assert len(spectra_files) == 1, 'Should only be 1 spectrum file per detection'
    spec_f_in = spectra_files[0]
    spec_f_out = spec_f_in.replace('.txt', '.fits')
    spec_f_cube = spec_f_in.replace('spec.txt', 'cube.fits')
    assert os.path.exists(spec_f_cube), 'Cutout cube corresponding to spectra file does not exist'
    row = write_table[write_table['name'] == source_name][0]
    spectrum_to_fits(spec_f_in, spec_f_cube, spec_f_out, row['ra'], row['dec'], row['freq'])

    # Rename all files
    all_files = glob.glob(os.path.join(pf_raw, '*'))
    renamed_files = []
    for af in all_files:
        fp = os.path.dirname(af)
        bn = os.path.basename(af)
        suffix = bn.rsplit('_', 1)[1]
        newname = os.path.join(fp, f'{source_name_file}_{suffix}')
        os.rename(af, newname)
        renamed_files.append(n)

In [ ]:
# Copy to VOSpace

vos_client = vos.Client()
for pf in product_files:
    files = glob.glob(os.path.join(pf, '*'))
    base_dir = os.path.basename(pf)
    
    # Should list files on VOS and skip if it exists
    
    try:
        vos_client.mkdir(f'vos:WALLABY/detections/{base_dir}')
    except Exception as e:
        print(e)
    for ff in files:
        basefilename = os.path.basename(ff)
        vos_client.copy(ff, f'vos:WALLABY/detections/{base_dir}/{basefilename}')
    print(f'Copied {base_dir}')

---